In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 24


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 3.3871522545814514
Epoch 2/100, Loss: 3.110112726688385
Epoch 3/100, Loss: 3.5756892189383507
Epoch 4/100, Loss: 3.4043615609407425
Epoch 5/100, Loss: 3.13773126155138
Epoch 6/100, Loss: 2.994248650968075
Epoch 7/100, Loss: 3.1750553622841835
Epoch 8/100, Loss: 3.437043257057667
Epoch 9/100, Loss: 3.120813399553299
Epoch 10/100, Loss: 3.7185876816511154
Epoch 11/100, Loss: 3.408920004963875
Epoch 12/100, Loss: 3.4017453864216805
Epoch 13/100, Loss: 3.564692460000515
Epoch 14/100, Loss: 3.223409526050091
Epoch 15/100, Loss: 3.5263360738754272
Epoch 16/100, Loss: 2.9409516379237175
Epoch 17/100, Loss: 3.6640551909804344


Epoch 18/100, Loss: 3.4472208470106125
Epoch 19/100, Loss: 3.27838134765625
Epoch 20/100, Loss: 3.482903391122818
Epoch 21/100, Loss: 3.5613007321953773
Epoch 22/100, Loss: 3.396634340286255
Epoch 23/100, Loss: 3.18790639936924
Epoch 24/100, Loss: 3.4604018926620483
Epoch 25/100, Loss: 3.339185245335102
Epoch 26/100, Loss: 3.534979499876499
Epoch 27/100, Loss: 3.6531809270381927
Epoch 28/100, Loss: 3.196453794836998
Epoch 29/100, Loss: 3.386891260743141
Epoch 30/100, Loss: 3.0168882608413696
Epoch 31/100, Loss: 3.225889764726162
Epoch 32/100, Loss: 3.742921195924282
Epoch 33/100, Loss: 3.1970741525292397
Epoch 34/100, Loss: 3.4376433193683624


Epoch 35/100, Loss: 3.2675831988453865
Epoch 36/100, Loss: 3.1935247629880905
Epoch 37/100, Loss: 3.1354561671614647
Epoch 38/100, Loss: 3.33726504445076
Epoch 39/100, Loss: 4.6093210726976395
Epoch 40/100, Loss: 3.241439498960972
Epoch 41/100, Loss: 3.58886232227087
Epoch 42/100, Loss: 3.6528222486376762
Epoch 43/100, Loss: 3.3455981612205505
Epoch 44/100, Loss: 3.4312222003936768
Epoch 45/100, Loss: 3.310652807354927
Epoch 46/100, Loss: 3.2272687926888466
Epoch 47/100, Loss: 3.176228918135166
Epoch 48/100, Loss: 3.519497886300087
Epoch 49/100, Loss: 3.612541474401951
Epoch 50/100, Loss: 3.259974353015423


Epoch 51/100, Loss: 3.2686657086014748
Epoch 52/100, Loss: 3.4269435554742813
Epoch 53/100, Loss: 3.2319383025169373
Epoch 54/100, Loss: 3.2640643790364265
Epoch 55/100, Loss: 2.8626462295651436
Epoch 56/100, Loss: 3.329521596431732
Epoch 57/100, Loss: 3.237585760653019
Epoch 58/100, Loss: 4.077956475317478
Epoch 59/100, Loss: 3.2497930228710175
Epoch 60/100, Loss: 3.228603795170784


Epoch 61/100, Loss: 3.391651064157486
Epoch 62/100, Loss: 3.6218282282352448
Epoch 63/100, Loss: 3.5137480050325394
Epoch 64/100, Loss: 3.5873124822974205
Epoch 65/100, Loss: 3.362672232091427
Epoch 66/100, Loss: 3.531105153262615
Epoch 67/100, Loss: 3.339432343840599
Epoch 68/100, Loss: 3.28959359228611
Epoch 69/100, Loss: 3.180084355175495
Epoch 70/100, Loss: 3.461190938949585


...

Epoch 30/100, Loss: 2.0040870308876038
Epoch 31/100, Loss: 2.0013138577342033
Epoch 32/100, Loss: 2.020134150981903
Epoch 33/100, Loss: 1.968207836151123
Epoch 34/100, Loss: 2.005934461951256
Epoch 35/100, Loss: 1.9408423751592636
Epoch 36/100, Loss: 1.9378948211669922
Epoch 37/100, Loss: 1.9995232298970222
Epoch 38/100, Loss: 2.0767873004078865
Epoch 39/100, Loss: 2.0712301209568977
Epoch 40/100, Loss: 1.9418208301067352
Epoch 41/100, Loss: 2.0159241184592247
Epoch 42/100, Loss: 1.9375062957406044
Epoch 43/100, Loss: 1.9616874232888222
Epoch 44/100, Loss: 2.2624702900648117
Epoch 45/100, Loss: 1.9615385830402374


Epoch 46/100, Loss: 2.0463357493281364
Epoch 47/100, Loss: 2.021151728928089
Epoch 48/100, Loss: 2.0300913751125336
Epoch 49/100, Loss: 1.953770138323307
Epoch 50/100, Loss: 2.005867585539818
Epoch 51/100, Loss: 2.132169984281063
Epoch 52/100, Loss: 1.9421334639191628
Epoch 53/100, Loss: 1.9314763247966766
Epoch 54/100, Loss: 1.9872559905052185
Epoch 55/100, Loss: 2.090036928653717
Epoch 56/100, Loss: 1.9254038482904434
Epoch 57/100, Loss: 1.9145777598023415
Epoch 58/100, Loss: 1.9764912948012352
Epoch 59/100, Loss: 1.9225260019302368
Epoch 60/100, Loss: 1.9930684119462967
Epoch 61/100, Loss: 1.9785223230719566
Epoch 62/100, Loss: 2.153491973876953


Epoch 63/100, Loss: 2.1413977295160294
Epoch 64/100, Loss: 1.975173905491829
Epoch 65/100, Loss: 1.9341546893119812
Epoch 66/100, Loss: 2.0437394604086876
Epoch 67/100, Loss: 2.1116612628102303
Epoch 68/100, Loss: 2.036270745098591
Epoch 69/100, Loss: 1.9741676226258278
Epoch 70/100, Loss: 1.8991370648145676
Epoch 71/100, Loss: 1.9943536594510078
Epoch 72/100, Loss: 2.031346894800663
Epoch 73/100, Loss: 2.009469397366047
Epoch 74/100, Loss: 1.955900028347969
Epoch 75/100, Loss: 1.9927426800131798
Epoch 76/100, Loss: 2.1703555434942245
Epoch 77/100, Loss: 2.0413437709212303


Epoch 78/100, Loss: 2.002725824713707
Epoch 79/100, Loss: 2.052963376045227
Epoch 80/100, Loss: 2.0422235652804375
Epoch 81/100, Loss: 2.036149024963379
Epoch 82/100, Loss: 1.964496023952961
Epoch 83/100, Loss: 1.8979908898472786
Epoch 84/100, Loss: 2.024006024003029
Epoch 85/100, Loss: 1.9163884073495865
Epoch 86/100, Loss: 1.927446387708187
Epoch 87/100, Loss: 2.044370822608471
Epoch 88/100, Loss: 2.020404174923897
Epoch 89/100, Loss: 2.00876072794199
Epoch 90/100, Loss: 1.9506068155169487
Epoch 91/100, Loss: 2.069646269083023
Epoch 92/100, Loss: 1.9397407844662666


Epoch 93/100, Loss: 2.176242917776108
Epoch 94/100, Loss: 1.9726872369647026
Epoch 95/100, Loss: 2.0470929369330406
Epoch 96/100, Loss: 1.9964414089918137
Epoch 97/100, Loss: 2.018549233675003
Epoch 98/100, Loss: 2.02573125064373
Epoch 99/100, Loss: 2.059466354548931
Epoch 100/100, Loss: 1.9373375251889229
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6816
